# GeoMamba — Slotlama Kontrolleri: kazanç geometrik mi? (Kaggle T4)

Ön-kayıt ve karar kuralları: `report/preregistration_slot_controls.md`
(koşumdan **önce** yazıldı, sonuçlara göre değiştirilmeyecek).

5 gömme kolu × 2 mimari × 5 seed = **50 koşu**, D1 rijit cisim, N=300.
Her koşu ayrıca döndürülmüş test setinde de skorlanır (`--rot-test`) —
eşdeğişkenlik probu, ek eğitim maliyeti yok.

| kol | ne yapar |
|---|---|
| `free` | serbest Linear gömme (baseline) |
| `slot2` | ω → grade-2 bıçakları (mevcut kazanan) |
| `slot1` | ω → grade-1 bıçakları — **fark çıkmamalı** |
| `scatter` | skaler + tek vektör bıçağı + pseudoskaler — aynı parametre, geometri yok |
| `frozen` | serbest Linear, dondurulmuş — "öğrenecek bir şey yok" kontrolü |

**Kurulum (bir kere):**
1. Sağdaki panel → **Session options** → Accelerator: **GPU T4 x2**, Internet: **On**
2. Üst menü → **Add-ons → Secrets** → `GH_TOKEN` ekleyin (repo `geomamba`, Contents: Read-only).
3. **Save Version → Save & Run All (Commit)** ile çalıştırın.

**ÖNEMLİ — kalıcılık:** İnteraktif Run All ile DEĞİL, **Save & Run All** ile
çalıştırın; interaktif session kapanınca çıktılar silinir.

**Süre:** T4 x2'de ~2 saat. İlk hücrelerdeki veri üretimi birkaç dakika
sessiz kalır (CPU'da 3000 yörünge integrasyonu) — donmuş sanıp durdurmayın.


In [ ]:
%cd /kaggle/working
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf /kaggle/working/geomamba
!git clone --depth 1 https://{token}@github.com/yusuftiryaki/geomamba.git /kaggle/working/geomamba


In [ ]:
%cd /kaggle/working/geomamba
import os, torch
os.environ["PYTHONPATH"] = "/kaggle/working/geomamba/src"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Session options'tan GPU secin!")


In [ ]:
import os, subprocess, sys, pathlib
GRID = "configs/grid/m5_slot_controls.yaml"

n_gpu = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU "))
print("GPU sayisi:", n_gpu)

# datasets once, then one process per GPU
subprocess.run([sys.executable, "scripts/run_grid.py", "--grid", GRID, "--data-only"], check=True)
procs = []
for k in range(max(n_gpu, 1)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(f"shard{k}.log", "w")
    procs.append((k, log, subprocess.Popen(
        [sys.executable, "scripts/run_grid.py", "--grid", GRID, "--skip-data",
         "--shard", f"{k}/{max(n_gpu, 1)}"],
        env=env, stdout=log, stderr=subprocess.STDOUT)))
print(f"{len(procs)} shard baslatildi; loglar shard*.log")
for k, log, p in procs:
    rc = p.wait(); log.close()
    print(f"shard {k} bitti, rc={rc}")
    print(pathlib.Path(f"shard{k}.log").read_text()[-2000:])


In [ ]:
import glob, shutil, pandas as pd
# Shard sonuçları iki yere yazılır: koşu sırasında artımlı olarak
# /kaggle/working/grid_results-shard*.csv, shard TAMAMEN bitince de
# experiments/ altına. Shard zaman aşımına uğrarsa ikincisi hiç yazılmaz --
# o yüzden ikisine de bakıyoruz, yoksa tamamlanmış koşular kayıp görünür.
paths = sorted(set(glob.glob("experiments/grid_results*.csv")
                   + glob.glob("/kaggle/working/grid_results-shard*.csv")))
frames = []
for f in paths:
    try:
        frames.append(pd.read_csv(f))
    except Exception as e:
        print("okunamadi", f, e)
if frames:
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset="name", keep="last")
    df.to_csv("/kaggle/working/grid_results.csv", index=False)
    for f in glob.glob("experiments/grid_results*.json"):
        shutil.copy(f, "/kaggle/working/")
    print(f"{len(df)} kosu toplandi ({len(paths)} dosyadan)")
    display(df)
else:
    print("sonuc yok - shard*.log dosyalarina bakin")
    for f in sorted(glob.glob("shard*.log")):
        print("=" * 20, f)
        print(open(f).read()[-3000:])


In [ ]:
# Ön-kayıttaki karar kurallarını uygula (report/preregistration_slot_controls.md)
import os, pandas as pd
if not os.path.exists("/kaggle/working/grid_results.csv"):
    raise SystemExit("Sonuc dosyasi yok - onceki hucrenin ciktisina bakin; "
                     "shard'lar cokmus ya da hic kosu bitmemis olabilir.")
df = pd.read_csv("/kaggle/working/grid_results.csv")
df["arch"] = df.name.str.split("-").str[1]
df["arm"]  = df.name.str.split("-").str[2]
cols = ["rollout_mse_h100", "rollout_mse_h50", "energy_drift", "rot_gap", "rot_gap_h100"]
piv = df.groupby(["arch", "arm"])[cols].agg(["mean", "std"]).round(4)
display(piv)
print("\npersistence_rot_gap (normalizasyon tabanı):", round(df.persistence_rot_gap.mean(), 3))

# eşleştirilmiş seed karşılaştırmaları: kaç seed'de A < B ?
def duel(a, b, arch, metric="rollout_mse_h100"):
    x = df[(df.arch == arch) & (df.arm == a)].sort_values("name")[metric].values
    y = df[(df.arch == arch) & (df.arm == b)].sort_values("name")[metric].values
    n = min(len(x), len(y))
    return f"{a} < {b}: {int((x[:n] < y[:n]).sum())}/{n}  ({x[:n].mean():.3f} vs {y[:n].mean():.3f})"

for arch in ("mamba", "rotor"):
    print(f"\n--- {arch} ---")
    for a, b in [("slot2","free"),("slot2","scatter"),("slot2","frozen"),
                 ("slot2","slot1"),("scatter","free"),("frozen","free")]:
        print(" ", duel(a, b, arch))
